# A Strands agent on Amazon Bedrock AgentCore that acts for the signed-in person

A person signs in to your app with Okta. Your Strands agent, deployed on Amazon Bedrock AgentCore,
then works **for that person**. It exchanges the person's Okta ID token for Highflame tokens, and
sends all its traffic through Highflame:

| Traffic | Highflame gateway | What Highflame does |
| --- | --- | --- |
| The model calls (Bedrock) | LLM gateway | Records each call, and checks every tool call the model makes **before the agent runs it**. |
| The tool calls (MCP) | MCP gateway | Calls the MCP server with **the person's own connection**, and checks each call against your policies. |

Every recorded call names the person and the agent, and one gateway policy applies on both
paths.

> **Customer-managed deployments only.** Highflame links the Okta person to a member of your
> organization by a verified email on one of your claimed domains. That needs a deployment
> with domain join. Highflame's hosted service does not offer it yet.

| Part | What it shows | Where it runs |
| --- | --- | --- |
| A | The person's token: sign-in, then the exchange | Here |
| B | The agent ([`agent_app.py`](agent_app.py)): a Strands agent packaged as an AgentCore app | On this machine, called the way AgentCore calls it |
| C | What Highflame records and enforces: the person's own connection, and a rule that blocks a tool | Here |
| D | Deploy the same agent to AgentCore | Your AWS account |

The standards: OpenID Connect sign-in, OAuth 2.0 Token Exchange
([RFC 8693](https://www.rfc-editor.org/rfc/rfc8693)) with `private_key_jwt`
([RFC 7523](https://www.rfc-editor.org/rfc/rfc7523)), and Resource Indicators
([RFC 8707](https://www.rfc-editor.org/rfc/rfc8707)) that bind each token to one gateway.

## Setup

Do the one-time setup in the [README](README.md):
[Set it up in Highflame](README.md#set-it-up-in-highflame) (steps 1 to 5) and
[the agent's app](README.md#the-agents-app-both-flows) in Okta. Then:

1. **An MCP server that people sign in to one by one.** Register it at your Highflame MCP gateway
   with per-person sign-in (OAuth). The person connects their own account once: in Studio →
   **AI Gateway** → **MCP Servers**, open the server and choose **Connect with OAuth**. This
   notebook uses Notion.
2. **AWS Bedrock in Highflame.** In Studio → **AI Gateway** → **LLM Providers** → **AWS Bedrock** →
   **Configure**, set the Bedrock endpoint of your region, for example
   `https://bedrock-runtime.us-east-1.amazonaws.com`, and save. The provider is bring-your-own-key:
   Highflame stores no AWS secret.
3. **AWS credentials** that may call the Bedrock model, for example after `aws sso login`.

The person must be a member of your Highflame organization, with the same email as in Okta.

Copy `.env.example` to `.env`, beside this notebook, and fill it in:

| Setting | What it is |
| --- | --- |
| `HIGHFLAME_IDENTITY_URL` | Your deployment's identity endpoint. |
| `MCP_SERVER_URL` | The MCP server, at your Highflame MCP gateway. |
| `HIGHFLAME_LLM_URL` | The LLM gateway: the **LLM Base URL** on the LLM Providers page, followed by `/v1`. |
| `AGENT_PRIVATE_KEY_PATH` | The agent's delegation key: the private half of the key on the agent in Studio. |
| `OKTA_ORG_URL`, `OKTA_CLIENT_ID` | Your Okta org, and the client id of the agent's app. |
| `OKTA_CLIENT_SECRET`, or `OKTA_PRIVATE_KEY_PATH` and `OKTA_KEY_ID` | How the agent's app authenticates to Okta. |
| `AWS_REGION`, `AWS_PROFILE` (optional) | Where Bedrock runs, and the AWS profile to use. |
| `BEDROCK_MODEL_ID` (optional) | The default is `us.anthropic.claude-sonnet-4-5-20250929-v1:0`. |
| `SCOPE` (optional) | The scopes the agent asks for. The default is `tools:read`. |

In [ ]:
%pip install -q -r requirements.txt

from importlib.metadata import version

from highflame.zeroid.resources.tokens import TokensResource

if not hasattr(TokensResource, "exchange_id_token"):
    raise RuntimeError(
        f"highflame {version('highflame')} has no tokens.exchange_id_token(). This notebook needs"
        " highflame 0.6.0 or later. Re-run the install cell above, then restart the kernel."
    )
for package in ["highflame", "strands-agents", "bedrock-agentcore", "mcp"]:
    print(f"{package:<20} {version(package)}")

In [ ]:
import os

# common.py loads .env from this folder. Real environment variables win.
from common import (agent_private_key, highflame_tokens, mcp_server_url, saved_id_token, scope,
                    setting, show_claims)
from signin import sign_in

for name in ["HIGHFLAME_IDENTITY_URL", "MCP_SERVER_URL", "HIGHFLAME_LLM_URL", "AGENT_PRIVATE_KEY_PATH",
             "OKTA_ORG_URL", "OKTA_CLIENT_ID", "AWS_REGION"]:
    print(f"{name:<24} {'set' if os.environ.get(name) else 'MISSING: set it in .env'}")

import boto3

try:
    boto3.client("sts").get_caller_identity()
    print(f"{'AWS credentials':<24} found")
except Exception as e:  # noqa: BLE001 (any failure means: no usable credentials)
    print(f"{'AWS credentials':<24} MISSING ({type(e).__name__}): sign in to AWS, for example aws sso login")

## Part A. The person's token

### A1. The person signs in

This is the agent's ordinary Okta sign-in, and nothing in it is specific to Highflame. In
production your app does it, and sends the person's ID token to the agent with each request.

The cell reuses the saved sign-in in `.tokens/` while its ID token is fresh, then refreshes it with
the refresh token. Highflame accepts an ID token for 10 minutes after Okta issues it, unless your
trusted issuer sets another limit. With no usable sign-in, the cell opens Okta's sign-in page: sign
in **as the person**, in a browser on this machine.

In [ ]:
id_token = saved_id_token() or sign_in()
if not id_token:
    raise RuntimeError("The person did not sign in. Run this cell again.")

person = show_claims("The person's ID token, from Okta", id_token,
                     ["iss", "sub", "aud", "email", "email_verified"])

`aud` is the agent's Okta app. Okta leaves `email_verified` out of its ID tokens. The issuer's
**Trust this directory's email addresses** setting covers that, so Highflame can link the person
by their email the first time it sees them.

### A2. The agent exchanges the ID token, once per gateway

The agent calls Highflame's token endpoint with the person's ID token, signed with its delegation
key. It asks for two tokens, each bound (`resource`) to the one gateway that receives it. The agent
in Part B does exactly this on every request.

In [ ]:
from highflame.zeroid import ZeroIDError

LLM_URL = setting("HIGHFLAME_LLM_URL")


def exchange(resource: str) -> str:
    try:
        return highflame_tokens().exchange_id_token(
            id_token,
            client_id=setting("OKTA_CLIENT_ID"),  # the ID token's aud, linked to the agent in Studio
            private_key_pem=agent_private_key(),   # private_key_jwt with the delegation key
            scope=scope(),
            resource=resource,                     # RFC 8707: this token works at this place only
        ).access_token
    except ZeroIDError as e:
        raise RuntimeError(f"Highflame refused the exchange ({e.status} {e.code}): {e.message}") from None


mcp_token = exchange(mcp_server_url())
llm_token = exchange(LLM_URL)
hf = show_claims("The token for the MCP gateway", mcp_token,
                 ["sub", "principal_type", "act.sub", "aud", "scope", "user_idp_sub"])
show_claims("The token for the LLM gateway", llm_token, ["sub", "act.sub", "aud"]);

| Claim | Meaning |
| --- | --- |
| `sub` | The person's **Highflame** user id, not their Okta id. Highflame found the member with the same email, on a claimed domain, and linked the two. |
| `principal_type` | `user`: the token stands for a person. |
| `act.sub` | The agent that acts for them. |
| `aud` | The one gateway this token is for. |
| `scope` | The request, inside the agent's credential policy. |
| `user_idp_sub` | The person's Okta id, kept for audit. |

## Part B. The Strands agent, as an AgentCore app

[`agent_app.py`](agent_app.py) is a Strands agent packaged with `BedrockAgentCoreApp`, the same
code that Part D deploys. For each request it:

1. reads the person's ID token from `Authorization`. On AgentCore, the runtime validates it against
   Okta first, and passes it on because `Authorization` is on its request header allowlist;
2. exchanges it twice, as in A2;
3. builds the model: Bedrock **through the Highflame LLM gateway**. The short-lived Bedrock API key
   goes in `Authorization`, and the person's token in `X-Highflame-Token`;
4. connects to the MCP server **through the Highflame MCP gateway** with the person's token, and
   gives the agent the server's read-only tools;
5. runs the prompt, and returns the answer and the tools that ran.

The next cell starts it on this machine, on port 8080, the way AgentCore runs it.

In [ ]:
import subprocess
import sys
import time

import httpx

AGENT_URL = "http://localhost:8080"
app_process = subprocess.Popen([sys.executable, "agent_app.py"],
                               stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        if httpx.get(f"{AGENT_URL}/ping", timeout=2).is_success:
            break
    except httpx.TransportError:
        pass
    time.sleep(1)
else:
    raise RuntimeError("The agent did not start. Run `python agent_app.py` in a terminal to see why.")
print("agent running:", httpx.get(f"{AGENT_URL}/ping").json())

Now call it the way AgentCore calls it: `POST /invocations`, with the person's ID token as the
bearer.

In [ ]:
QUESTION = "Search my Notion workspace for 'roadmap' and give the titles of up to three pages, one line each."


def ask(prompt: str) -> dict:
    """One request to the agent, as the person."""
    reply = httpx.post(f"{AGENT_URL}/invocations", timeout=300, json={"prompt": prompt},
                       headers={"Authorization": f"Bearer {saved_id_token() or id_token}"}).json()
    if reply.get("error"):
        print("The agent refused:", reply["error"])
    else:
        print(reply["answer"])
        print("\ntools that ran:", reply.get("tools_called") or "none")
    return reply


ask(QUESTION);

The agent asked Bedrock through the LLM gateway, and called the Notion tool through the MCP
gateway with the person's own Notion connection. It never held a Notion credential.

## Part C. What Highflame sees and enforces

### C1. Every step names the person and the agent

Open Studio → **Observatory**, and find the request above. Each step is recorded:

| Span | From | What it holds |
| --- | --- | --- |
| `llm.route` | LLM gateway | One model call. `hf.user_id` is the person, `hf.agent_id` the agent. |
| `guard` without a server | LLM gateway | The check of a tool call **the model asked for**, before the agent runs it. |
| `mcp.tools.call` | MCP gateway | The tool call itself, sent to the MCP server. |
| `guard` with `hf.mcp_server_name` | MCP gateway | The check of that call before it runs. |

On each `guard` span, `hf.tool_name` is the tool, `hf.decision` is `allow` or `deny`, and
`hf.act_sub` is the person.

### C2. The person's own connection

For an MCP server that people sign in to one by one, the gateway uses the connection of the token's
`sub`, the person, never the agent's. So the agent can never do more there than the person can. If
the person has not connected the server, the gateway refuses every request with HTTP 403
`downstream_auth_required`. An MCP server set up with one shared credential has no per-person layer.

This call goes straight to the MCP gateway with the person's token, as the agent's tools do:

In [ ]:
from common import McpHttpError, McpSession

BLOCKED_TOOL = "notion-search"


def mcp_call(tool: str, arguments: dict) -> bool:
    """One tools/call at the MCP gateway as the person. True when Highflame refused it."""
    session = McpSession(mcp_server_url(), mcp_token)
    try:
        refused, text = session.call_tool(tool, arguments)
    except McpHttpError as e:
        if e.status == 403 and "downstream_auth_required" in e.body:
            print(f"{tool}: refused. The person has not connected this server: {e.body}")
            return True
        raise
    finally:
        session.close()
    print(f"{tool}: " + (f"REFUSED: {text}" if refused else f"ok, {len(text)} characters of results"))
    return refused


mcp_call(BLOCKED_TOOL, {"query": "roadmap"});

### C3. Block a tool, on both gateways

A gateway policy is Cedar. This one forbids one tool. Policies only narrow: Highflame's baseline
policy permits gateway calls, and a `forbid` takes one thing away.

The rule names no MCP server, so it applies at both gateways. It applies to every agent and every
person in the project, so try it in a test project.

In [ ]:
POLICY = f'''@id("gateway.forbid-{BLOCKED_TOOL}")
@severity("high")
@reject_message("{BLOCKED_TOOL} is not allowed in this project.")
forbid (
  principal,
  action == AIGateway::Action::"call_tool",
  resource == AIGateway::Tool::"{BLOCKED_TOOL}"
);'''
print(POLICY)

Add it in Studio:

1. Go to **AI Gateway** → **Policies**, and select the project of the MCP server.
2. Open the **Tool Permissioning** tab, and choose **Create Policy**.
3. Give the policy a name.
4. Set **Mode** to **Enforce**. A new policy starts in **Monitor**, which only records.
5. Choose the **Cedar editor** button (`</>`, marked Beta) at the top of the editor. Replace the
   text with the policy above.
6. Choose **Deploy**.

The gateways apply a new policy within about 30 seconds. The next cell waits for it, then asks the
agent the same question again.

In [ ]:
deadline = time.time() + 90
while not (refused := mcp_call(BLOCKED_TOOL, {"query": "roadmap"})) and time.time() < deadline:
    time.sleep(10)
if not refused:
    print("Not refused yet. Check that the policy is deployed, in Enforce mode, in the project of the MCP server.")

print()
ask(QUESTION);

The LLM gateway refused the model's `notion-search` call before the agent could run it: no tool
ran, and the agent reports the refusal. Without the LLM gateway, the MCP gateway would have refused
the call one step later.

To make the rule apply at the MCP gateway only, add `when { context has mcp_server &&
context.mcp_server == "notion" }`: the LLM gateway does not know which server a tool belongs to.

When you are done, turn the policy's **Active** switch off, or delete it. Then stop the local agent:

In [ ]:
app_process.terminate()
app_process.wait(timeout=10)
print("agent stopped")

## Part D. Deploy the same agent to AgentCore

The agent code does not change. On AgentCore:

| On this machine | On AgentCore |
| --- | --- |
| You send the ID token in `Authorization` | Your app sends it. The runtime's JWT authorizer validates it against Okta, and forwards it because `Authorization` is on the request header allowlist. |
| `AGENT_PRIVATE_KEY_PATH` | `AGENT_PRIVATE_KEY_SECRET_ID`: the delegation key in AWS Secrets Manager. |
| Your AWS credentials make the Bedrock API key | The runtime's execution role does. |
| `localhost` URLs | Your Highflame deployment's URLs. They must be reachable from AWS. |

The execution role needs `secretsmanager:GetSecretValue` on the key's secret, and
`bedrock:InvokeModel`, `bedrock:InvokeModelWithResponseStream` and `bedrock:CallWithBearerToken` for
the model. Install the deploy tool with `pip install bedrock-agentcore-starter-toolkit`.

The cells below do nothing until you set `DEPLOY = True`.

In [ ]:
DEPLOY = False  # set to True to run the deploy cells against your AWS account

AGENT_NAME = "highflame_acts_for_person"
REGION = os.environ.get("AWS_REGION", "us-east-1")
SECRET_NAME = f"{AGENT_NAME}/delegation-key"

if DEPLOY:
    secrets = boto3.client("secretsmanager", region_name=REGION)
    secret = secrets.create_secret(Name=SECRET_NAME, SecretString=agent_private_key())
    print("delegation key stored:", secret["ARN"])

Configure the runtime: the entrypoint, an Okta JWT authorizer, and the header allowlist. An Okta ID
token's `aud` is the client id of the agent's app, so the authorizer checks `allowedAudience`.

In [ ]:
import json
import shlex

authorizer = {"customJWTAuthorizer": {
    "discoveryUrl": setting("OKTA_ORG_URL").rstrip("/") + "/.well-known/openid-configuration",
    "allowedAudience": [setting("OKTA_CLIENT_ID")],
}}
configure = ["agentcore", "configure", "--entrypoint", "agent_app.py", "--name", AGENT_NAME,
             "--requirements-file", "requirements.txt", "--region", REGION,
             "--authorizer-config", json.dumps(authorizer),
             "--request-header-allowlist", "Authorization"]
print(shlex.join(configure))
if DEPLOY:
    subprocess.run(configure, check=True)

Deploy it, with your Highflame deployment's URLs in place of the local ones:

In [ ]:
environment = {
    "HIGHFLAME_IDENTITY_URL": setting("HIGHFLAME_IDENTITY_URL"),
    "MCP_SERVER_URL": mcp_server_url(),
    "HIGHFLAME_LLM_URL": LLM_URL,
    "OKTA_CLIENT_ID": setting("OKTA_CLIENT_ID"),
    "AGENT_PRIVATE_KEY_SECRET_ID": SECRET_NAME,
    "AWS_REGION": REGION,
    "SCOPE": scope(),
}
deploy = ["agentcore", "deploy", "--agent", AGENT_NAME]
for key, value in environment.items():
    deploy += ["--env", f"{key}={value}"]
print(shlex.join(deploy))
if DEPLOY:
    subprocess.run(deploy, check=True)

Call it as the person, with their ID token as the bearer:

In [ ]:
invoke = ["agentcore", "invoke", json.dumps({"prompt": QUESTION}), "--agent", AGENT_NAME,
          "--bearer-token", saved_id_token() or id_token]
print(shlex.join(invoke[:-1]), "<the person's ID token>")
if DEPLOY:
    subprocess.run(invoke, check=True)

Gateway policies are made in Studio, by design: there is no API for creating them.

## Troubleshooting

See [Troubleshooting](README.md#troubleshooting) in the README. The most common problems:

| What you see | Why |
| --- | --- |
| `invalid_client` from the exchange | The agent's Okta app is not linked to the agent, or the delegation key does not match the key on the agent. |
| `invalid_grant` from the exchange | Highflame found no member for the person: no member with that email, the email's domain is not claimed, or the issuer is not verified. Or the ID token is too old: sign in again. |
| `unknown provider 'bedrock'` from the LLM gateway | AWS Bedrock is not configured under **AI Gateway** → **LLM Providers** (setup step 2). |
| 403 `downstream_auth_required` from the MCP gateway | The person has not connected the MCP server. |
| AgentCore answers 401 | The runtime's authorizer refused the token: check `allowedAudience` and the discovery URL. |
| The agent answers "No person" | `Authorization` is not on the runtime's request header allowlist. |